<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 50%, #1c2128 100%); padding: 45px 35px; border-radius: 20px; text-align: center; border: 1px solid #30363d; margin-bottom: 20px;">
  <h1 style="color: #58a6ff; font-size: 2.6em; margin: 0 0 12px 0; letter-spacing: 2px; font-family: 'Segoe UI', sans-serif;">🏆 Benchmarking des Algorithmes d'Apprentissage Supervisé</h1>
  <h2 style="color: #e94560; font-size: 1.4em; margin: 0 0 16px 0; font-weight: 400;">Classification du Type de Peau — Projet DeepSkyn</h2>
  <hr style="border: 1px solid #30363d; width: 65%; margin: 16px auto;"/>
  <p style="color: #8b949e; font-size: 1.0em; margin: 0;">Comparaison des performances : KNN · Decision Tree · Random Forest · SVM · XGBoost</p>
  <p style="color: #6e7681; font-size: 0.88em; margin-top: 10px;">Métriques : Accuracy · F1-Score (macro/weighted) · AUC-ROC · Temps de traitement</p>
</div>

In [ ]:
# ── Installation des dépendances (si nécessaire) ─────────────────────────────
import subprocess, sys
pkgs = ['seaborn', 'xgboost', 'scikit-learn', 'matplotlib', 'pandas', 'numpy']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ Toutes les dépendances sont disponibles.')

---
## 📦 0. Imports & Configuration

> Ce notebook agrège les résultats des 5 notebooks de modélisation individuels et effectue une comparaison exhaustive. Toutes les métriques sont extraites directement des sorties des cellules exécutées.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import time

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.preprocessing   import LabelEncoder, StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, f1_score
)
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.svm             import SVC
from xgboost                 import XGBClassifier

# ── Global Style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'        : 130,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 14,
    'axes.labelsize'    : 12,
    'xtick.labelsize'   : 11,
    'ytick.labelsize'   : 11,
    'legend.fontsize'   : 10,
    'font.family'       : 'DejaVu Sans',
})

# ── Palette de couleurs pour les 5 modèles ───────────────────────────────────
PALETTE = {
    'KNN'          : '#4361ee',
    'Decision Tree': '#f72585',
    'Random Forest': '#3a86ff',
    'SVM'          : '#ff6b35',
    'XGBoost'      : '#06d6a0',
}
MODEL_NAMES = list(PALETTE.keys())
MODEL_COLORS = list(PALETTE.values())

CLASS_NAMES = ['Combination', 'Dry', 'Normal', 'Oily']
N_CLASSES   = len(CLASS_NAMES)

print('✅ Configuration chargée — 5 algorithmes à comparer')
print(f'   Classes cibles : {CLASS_NAMES}')

---
## 📂 Chargement & Préprocessing des données

In [ ]:
# ── Chargement du dataset ─────────────────────────────────────────────────────
df = pd.read_csv('Skin_Type_dataset.csv')
print(f'Dataset chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

# ── Nettoyage ────────────────────────────────────────────────────────────────
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Après suppression doublons : {df.shape[0]:,} lignes')

# ── Encodage des variables catégorielles ─────────────────────────────────────
cat_cols = ['Gender', 'Hydration_Level', 'Oil_Level', 'Sensitivity']
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# ── Encodage de la cible ──────────────────────────────────────────────────────
le_target       = LabelEncoder()
df['Skin_Type'] = le_target.fit_transform(df['Skin_Type'])
CLASS_NAMES     = list(le_target.classes_)
print(f'Classes encodées : {CLASS_NAMES}')

# ── Features / Cible ─────────────────────────────────────────────────────────
FEATURES = ['Age', 'Gender', 'Hydration_Level', 'Oil_Level',
            'Sensitivity', 'Humidity', 'Temperature']

X = df[FEATURES]
y = df['Skin_Type']

# ── Découpage Train / Test ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Normalisation (nécessaire pour KNN et SVM) ───────────────────────────────
scaler            = StandardScaler()
X_train_sc        = scaler.fit_transform(X_train)
X_test_sc         = scaler.transform(X_test)

print(f'\nEnsemble d\'entraînement : {X_train.shape[0]:,} échantillons')
print(f'Ensemble de test        : {X_test.shape[0]:,} échantillons')
print(f'Features utilisées      : {FEATURES}')
print('\n✅ Préprocessing terminé — données prêtes pour le benchmarking.')

---
## 🔧 Étape 1 : Configurations Optimales des Hyperparamètres

> Les hyperparamètres ci-dessous ont été déterminés par **GridSearchCV** et **RandomizedSearchCV** avec validation croisée (3 à 5 plis) dans les notebooks individuels de chaque algorithme. Ils représentent la **configuration optimale** identifiée par la recherche automatique.

### 📋 Tableau Récapitulatif des Meilleurs Hyperparamètres

| Algorithme | Méthode d'Optimisation | Hyperparamètre Principal | Valeur Optimale | Autres Paramètres |
|:-----------|:-----------------------|:------------------------|:----------------|:------------------|
| **KNN** | GridSearchCV (cv=5) | `n_neighbors` | **17** | `metric=euclidean`, `weights=distance` |
| **Decision Tree** | CCP Post-Pruning + Pre-Pruning | `ccp_alpha` | **0.000944** | `max_depth=7`, `min_samples_split=10`, `min_samples_leaf=5` |
| **Random Forest** | GridSearchCV (cv=5, 36 combos) | `n_estimators` | **100–200** | `max_depth=None`, `max_features=sqrt`, `min_samples_split=2` |
| **SVM** | GridSearchCV (cv=3, 12 combos) | `kernel` | **rbf** | `C=1`, `gamma=0.1` |
| **XGBoost** | RandomizedSearchCV (cv=3, 40 iter) | `max_depth` | **7** | `learning_rate=0.3`, `n_estimators=100`, `subsample=0.8`, `gamma=1.0`, `reg_lambda=10`, `reg_alpha=0.1` |

In [ ]:
# ── Instanciation des 5 modèles avec leurs meilleurs hyperparamètres ─────────
# (Extraits des notebooks individuels exécutés)

models = {
    'KNN': KNeighborsClassifier(
        n_neighbors = 17,
        weights     = 'distance',
        metric      = 'euclidean'
    ),
    'Decision Tree': DecisionTreeClassifier(
        ccp_alpha         = 0.000944,
        max_depth         = 7,
        min_samples_split = 10,
        min_samples_leaf  = 5,
        random_state      = 42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators      = 100,
        max_depth         = None,
        max_features      = 'sqrt',
        min_samples_split = 2,
        random_state      = 42,
        n_jobs            = -1
    ),
    'SVM': SVC(
        kernel      = 'rbf',
        C           = 1,
        gamma       = 0.1,
        random_state= 42,
        probability = True   # nécessaire pour les courbes ROC
    ),
    'XGBoost': XGBClassifier(
        objective         = 'multi:softmax',
        num_class         = N_CLASSES,
        max_depth         = 7,
        learning_rate     = 0.3,
        n_estimators      = 100,
        subsample         = 0.8,
        colsample_bytree  = 0.5,
        gamma             = 1.0,
        reg_alpha         = 0.1,
        reg_lambda        = 10,
        min_child_weight  = 3,
        use_label_encoder = False,
        verbosity         = 0,
        random_state      = 42,
        n_jobs            = -1
    )
}

print('✅ 5 modèles instanciés avec leurs hyperparamètres optimaux :')
for name in models:
    print(f'   • {name}')

---
## 📊 Étape 2 : Benchmarking des Modèles (Obligatoire)

### 2A — Entraînement de tous les modèles & collecte des métriques

In [ ]:
# ── Entraînement et évaluation de chaque modèle ───────────────────────────────

results = {}

for name, model in models.items():
    # KNN et SVM utilisent les données normalisées
    if name in ('KNN', 'SVM'):
        X_tr, X_te = X_train_sc, X_test_sc
    else:
        X_tr, X_te = X_train.values, X_test.values

    # ──  Entraînement (mesure du temps) ──────────────────────────────────────
    t0 = time.time()
    model.fit(X_tr, y_train)
    train_time = time.time() - t0

    # ──  Prédictions ─────────────────────────────────────────────────────────
    t0 = time.time()
    y_pred = model.predict(X_te)
    infer_time = time.time() - t0

    # ──  Métriques de classification ─────────────────────────────────────────
    acc      = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_wtd   = f1_score(y_test, y_pred, average='weighted')
    f1_combo = f1_score(y_test, y_pred, average=None)[0]  # classe 'Combination'

    # ──  AUC-ROC (One-vs-Rest) ────────────────────────────────────────────────
    y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))
    if hasattr(model, 'predict_proba'):
        y_prob   = model.predict_proba(X_te)
        macro_auc = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='macro')
    else:
        y_prob    = None
        macro_auc = np.nan

    results[name] = {
        'model'      : model,
        'y_pred'     : y_pred,
        'y_prob'     : y_prob,
        'accuracy'   : acc,
        'f1_macro'   : f1_macro,
        'f1_weighted': f1_wtd,
        'f1_combo'   : f1_combo,
        'auc_macro'  : macro_auc,
        'train_time' : train_time,
        'infer_time' : infer_time,
    }

    bar = '█' * int(acc * 30)
    print(f'  {name:<15} | Acc={acc:.4f}  F1-macro={f1_macro:.4f}  AUC={macro_auc:.4f}  '
          f'Train={train_time:.2f}s  Inf={infer_time*1000:.1f}ms  |{bar}')

print('\n✅ Entraînement et évaluation de tous les modèles terminés.')

### 2B — Tableau comparatif synthétique

In [ ]:
# ── Construction du DataFrame de benchmarking ─────────────────────────────────

bench_df = pd.DataFrame([
    {
        'Modèle'              : name,
        'Accuracy (%)'        : round(r['accuracy'] * 100, 2),
        'F1-Score Macro'      : round(r['f1_macro'], 4),
        'F1-Score Weighted'   : round(r['f1_weighted'], 4),
        'F1 Combination'      : round(r['f1_combo'], 4),
        'AUC Macro (OvR)'     : round(r['auc_macro'], 4) if not np.isnan(r['auc_macro']) else '—',
        'Temps Entraîn. (s)'  : round(r['train_time'], 2),
        'Temps Inférence (ms)': round(r['infer_time'] * 1000, 1),
    }
    for name, r in results.items()
]).set_index('Modèle')

print('=' * 80)
print('  TABLEAU DE BENCHMARKING — Comparaison des 5 Algorithmes')
print('=' * 80)
display(bench_df)

# ── Identifier le meilleur modèle par métrique ────────────────────────────────
print('\n📌 Meilleur modèle par métrique :')
for col in ['Accuracy (%)', 'F1-Score Macro', 'F1-Score Weighted', 'F1 Combination']:
    best = bench_df[col].idxmax()
    val  = bench_df.loc[best, col]
    print(f'   • {col:<25} → {best} ({val})')

### 2C — Graphique comparatif des Accuracy (Bar Chart)

In [ ]:
# ── Bar Chart : Accuracy des 5 modèles ───────────────────────────────────────

names     = list(results.keys())
accs      = [results[n]['accuracy'] for n in names]
f1_macros = [results[n]['f1_macro'] for n in names]
f1_wtds   = [results[n]['f1_weighted'] for n in names]
colors    = [PALETTE[n] for n in names]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Benchmarking des Algorithmes — Métriques de Classification sur le Jeu de Test',
    fontsize=15, fontweight='bold', y=1.02
)

def make_bar(ax, values, title, color_list, ylabel='Score', ylim=(0.60, 0.92)):
    bars = ax.bar(names, values, color=color_list, edgecolor='white',
                  linewidth=1.4, width=0.6, zorder=3)
    ax.set_title(title, fontweight='bold', pad=12)
    ax.set_ylabel(ylabel)
    ax.set_ylim(*ylim)
    ax.grid(axis='y', alpha=0.3, zorder=0)
    ax.set_xticklabels(names, rotation=15, ha='right')
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{val:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold'
        )
    # Mettre en évidence le meilleur
    best_idx = int(np.argmax(values))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    ax.annotate(
        '🏆 Meilleur',
        xy=(bars[best_idx].get_x() + bars[best_idx].get_width() / 2,
            bars[best_idx].get_height() + 0.012),
        fontsize=9, ha='center', color='goldenrod', fontweight='bold'
    )

make_bar(axes[0], accs,      'Accuracy sur le Jeu de Test',    colors, 'Accuracy')
make_bar(axes[1], f1_macros, 'F1-Score Macro (non pondéré)',   colors, 'F1-Score')
make_bar(axes[2], f1_wtds,   'F1-Score Weighted (pondéré)',    colors, 'F1-Score')

plt.tight_layout()
plt.savefig('benchmarking_bar_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé : benchmarking_bar_metrics.png')

### 2D — Comparaison des F1-Scores par Classe (Focus : Combination)

In [ ]:
# ── F1-Score par classe pour chaque modèle ────────────────────────────────────

f1_per_class = np.array([
    f1_score(y_test, results[n]['y_pred'], average=None)
    for n in names
])

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle('F1-Score par Classe — Analyse Détaillée par Algorithme',
             fontsize=15, fontweight='bold', y=1.02)

# ── Panel 1 : Grouped bar ─────────────────────────────────────────────────────
x   = np.arange(N_CLASSES)
w   = 0.14
ax  = axes[0]
for i, (name, col) in enumerate(zip(names, colors)):
    offset = (i - 2) * w
    bars   = ax.bar(x + offset, f1_per_class[i], w, label=name, color=col,
                    edgecolor='white', linewidth=0.8, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.08)
ax.set_title('F1-Score par Classe et par Algorithme', fontweight='bold')
ax.legend(loc='upper right', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, zorder=0)
# Annotation classe difficile
ax.annotate('⚠️ Classe\ndifficile', xy=(0, 0.36), fontsize=9,
            ha='center', color='#e94560', fontweight='bold')

# ── Panel 2 : Heatmap ─────────────────────────────────────────────────────────
hm_df = pd.DataFrame(
    f1_per_class,
    index   = names,
    columns = CLASS_NAMES
)

sns.heatmap(
    hm_df,
    annot      = True,
    fmt        = '.3f',
    cmap       = 'RdYlGn',
    vmin       = 0.0,
    vmax       = 1.0,
    linewidths = 0.5,
    ax         = axes[1],
    annot_kws  = {'size': 12, 'weight': 'bold'}
)
axes[1].set_title('Heatmap F1-Score : Modèle × Classe', fontweight='bold')
axes[1].set_xlabel('Classe (Type de Peau)')
axes[1].set_ylabel('Algorithme')

plt.tight_layout()
plt.savefig('benchmarking_f1_per_class.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé : benchmarking_f1_per_class.png')

### 2E — Courbes ROC Unifiées (One-vs-Rest, tous les modèles)

In [ ]:
# ── Courbes ROC unifiées ──────────────────────────────────────────────────────
# Une courbe ROC macro-AUC par modèle (agrégée par classe)

y_test_bin = label_binarize(y_test, classes=list(range(N_CLASSES)))

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.suptitle('Courbes ROC — Comparaison de tous les Algorithmes (One-vs-Rest)',
             fontsize=15, fontweight='bold', y=1.01)

# ── Panel gauche : Macro AUC par modèle (1 courbe par algo) ──────────────────
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Classifieur aléatoire (AUC=0.5)')

for name, col in zip(names, colors):
    if results[name]['y_prob'] is None:
        continue
    y_prob = results[name]['y_prob']
    # Calcul de la courbe ROC macro-moyennée
    all_fpr = np.unique(np.concatenate([
        roc_curve(y_test_bin[:, i], y_prob[:, i])[0]
        for i in range(N_CLASSES)
    ]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(N_CLASSES):
        fpr_i, tpr_i, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
    mean_tpr /= N_CLASSES
    macro_auc_val = auc(all_fpr, mean_tpr)
    ax.plot(all_fpr, mean_tpr, color=col, linewidth=2.5,
            label=f'{name}  (AUC = {macro_auc_val:.4f})')

ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
ax.set_title('ROC Macro-Moyennée — Tous les Algorithmes', fontweight='bold')
ax.legend(loc='lower right', framealpha=0.95)
ax.grid(alpha=0.3)

# ── Panel droit : AUC par modèle (bar chart) ─────────────────────────────────
ax2     = axes[1]
auc_vals = [results[n]['auc_macro'] for n in names]
bars     = ax2.barh(names, auc_vals, color=colors, edgecolor='white',
                    linewidth=1.4, height=0.5, zorder=3)
ax2.set_xlim(0.6, 1.0)
ax2.set_xlabel('AUC-ROC Macro', fontsize=12)
ax2.set_title('AUC-ROC Macro par Algorithme', fontweight='bold')
ax2.grid(axis='x', alpha=0.3, zorder=0)
ax2.invert_yaxis()
for bar, val in zip(bars, auc_vals):
    ax2.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=11, fontweight='bold')
# Mettre en évidence le meilleur
best_idx = int(np.argmax(auc_vals))
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)

plt.tight_layout()
plt.savefig('benchmarking_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Courbes ROC sauvegardées : benchmarking_roc_curves.png')

### 2F — Matrices de Confusion (comparaison côte à côte)

In [ ]:
# ── Matrices de confusion normalisées pour les 5 modèles ─────────────────────

fig, axes = plt.subplots(1, 5, figsize=(24, 5))
fig.suptitle('Matrices de Confusion Normalisées — Comparaison des 5 Algorithmes',
             fontsize=15, fontweight='bold', y=1.02)

cmaps = ['Blues', 'RdPu', 'Greens', 'Oranges', 'YlGn']

for ax, (name, col), cmap in zip(axes, zip(names, colors), cmaps):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(
        cm_norm,
        annot       = True,
        fmt         = '.2f',
        cmap        = cmap,
        vmin        = 0,
        vmax        = 1,
        ax          = ax,
        xticklabels = [c[:4] for c in CLASS_NAMES],
        yticklabels = [c[:4] for c in CLASS_NAMES],
        linewidths  = 0.5,
        cbar        = False,
        annot_kws   = {'size': 9, 'weight': 'bold'}
    )
    ax.set_title(f'{name}\nAcc = {results[name]["accuracy"]:.3f}',
                 fontweight='bold', color=col, fontsize=12)
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.tight_layout()
plt.savefig('benchmarking_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Matrices de confusion sauvegardées : benchmarking_confusion_matrices.png')

### 2G — Analyse des Temps de Calcul (Performance vs Efficacité)

In [ ]:
# ── Comparaison vitesse vs précision ─────────────────────────────────────────

train_times  = [results[n]['train_time']     for n in names]
infer_times  = [results[n]['infer_time']*1000 for n in names]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Analyse des Temps de Calcul — Entraînement & Inférence',
             fontsize=15, fontweight='bold', y=1.02)

# Panel 1 : Temps d'entraînement
bars1 = axes[0].bar(names, train_times, color=colors, edgecolor='white',
                    linewidth=1.4, width=0.6, zorder=3)
axes[0].set_title('Temps d\'Entraînement (secondes)', fontweight='bold')
axes[0].set_ylabel('Temps (s)')
axes[0].grid(axis='y', alpha=0.3, zorder=0)
axes[0].set_xticklabels(names, rotation=15, ha='right')
for bar, val in zip(bars1, train_times):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}s', ha='center', fontsize=10, fontweight='bold')

# Panel 2 : Temps d'inférence
bars2 = axes[1].bar(names, infer_times, color=colors, edgecolor='white',
                    linewidth=1.4, width=0.6, zorder=3)
axes[1].set_title('Temps d\'Inférence sur 2000 échantillons (ms)', fontweight='bold')
axes[1].set_ylabel('Temps (ms)')
axes[1].grid(axis='y', alpha=0.3, zorder=0)
axes[1].set_xticklabels(names, rotation=15, ha='right')
for bar, val in zip(bars2, infer_times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}ms', ha='center', fontsize=10, fontweight='bold')

# Panel 3 : Scatter Accuracy vs Temps d'entraînement
for i, (name, col) in enumerate(zip(names, colors)):
    axes[2].scatter(
        train_times[i], accs[i],
        s=250, color=col, edgecolors='white', linewidth=1.5,
        label=name, zorder=4
    )
    axes[2].annotate(
        name,
        (train_times[i], accs[i]),
        textcoords='offset points',
        xytext=(8, 5),
        fontsize=9, fontweight='bold'
    )

axes[2].set_xlabel('Temps d\'Entraînement (s)', fontsize=12)
axes[2].set_ylabel('Accuracy', fontsize=12)
axes[2].set_title('Trade-off : Précision vs Temps de Calcul', fontweight='bold')
axes[2].grid(alpha=0.3)
axes[2].set_xscale('log')

# Annotation zones
axes[2].annotate('⭐ Zone idéale\n(Rapide + Précis)',
                 xy=(0.02, 0.82), xycoords='axes fraction',
                 fontsize=9, color='green', fontstyle='italic')

plt.tight_layout()
plt.savefig('benchmarking_time_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Analyse temporelle sauvegardée : benchmarking_time_analysis.png')

### 2H — Radar Chart : Vue d'ensemble multi-dimensionnelle

In [ ]:
# ── Radar Chart (Spider Plot) — performances multi-dimensionnelles ─────────────

categories  = ['Accuracy', 'F1 Macro', 'F1 Weighted', 'F1 Combination', 'AUC Macro']
N_CAT       = len(categories)
angles      = np.linspace(0, 2 * np.pi, N_CAT, endpoint=False).tolist()
angles     += angles[:1]  # fermer le cercle

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'polar': True})
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f0f2f5')

for name, col in zip(names, colors):
    r = results[name]
    vals = [
        r['accuracy'],
        r['f1_macro'],
        r['f1_weighted'],
        r['f1_combo'],
        r['auc_macro'] if not np.isnan(r['auc_macro']) else 0.0
    ]
    vals += vals[:1]
    ax.plot(angles, vals, color=col, linewidth=2.5, linestyle='solid', label=name)
    ax.fill(angles, vals, color=col, alpha=0.10)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=9)
ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_title('Profil Multi-Dimensionnel des 5 Algorithmes',
             fontsize=14, fontweight='bold', pad=25, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.32, 1.15), framealpha=0.95,
          fontsize=11, title='Algorithmes', title_fontsize=11)

plt.tight_layout()
plt.savefig('benchmarking_radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Radar chart sauvegardé : benchmarking_radar_chart.png')

### 2I — Rapports de Classification Complets

In [ ]:
# ── Rapports de classification pour chaque modèle ────────────────────────────

for name in names:
    print('\n' + '=' * 58)
    print(f'  RAPPORT DE CLASSIFICATION — {name}')
    print('=' * 58)
    print(classification_report(
        y_test, results[name]['y_pred'],
        target_names=CLASS_NAMES,
        digits=4
    ))

---
## 💬 Étape 3 : Discussion sur les Forces et Faiblesses des Algorithmes

<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 100%); padding: 30px; border-radius: 14px; border: 1px solid #30363d; color: #c9d1d9; font-family: 'Segoe UI', sans-serif;">

---

### 🔵 K-Nearest Neighbors (KNN)

**Configuration optimale :** `n_neighbors=17`, `weights=distance`, `metric=euclidean`  
**Performance :** Accuracy ≈ 77.6% | F1-macro ≈ 0.68

**Forces :**
- Algorithme non-paramétrique, aucune hypothèse sur la distribution des données.
- Simple à comprendre et à implémenter, bon point de départ pour le benchmarking.
- Performant sur des espaces de features bien normalisées.

**Faiblesses :**
- **Scalabilité limitée** : le temps d'inférence croît linéairement avec la taille du dataset (O(n)).
- Très sensible au **bruit** et aux **outliers** dans les données.
- La **malédiction de la dimensionnalité** le pénalise pour les datasets à haute dimension.
- Nécessite une **normalisation systématique** (StandardScaler) pour fonctionner correctement, car il est basé sur des distances euclidiennes.
- Performance médiocre sur la classe `Combination` (F1 ≈ 0.17), la plus difficile du dataset.

---

### 🌸 Decision Tree (Arbre de Décision)

**Configuration optimale :** `ccp_alpha=0.000944`, `max_depth=7`, `min_samples_leaf=5`  
**Performance :** Accuracy ≈ 79.3% | F1-macro ≈ 0.72

**Forces :**
- **Interprétabilité maximale** : l'arbre peut être visualisé, chaque règle de décision est explicable.
- Résistant aux features non normalisées, aucune mise à l'échelle requise.
- **Rapide à entraîner** sur des datasets tabullaires de taille modérée.
- L'élagage ( *pruning*) par CCP permet de contrôler efficacement le sur-apprentissage.

**Faiblesses :**
- **Instabilité** : une faible variation des données peut entraîner une structure d'arbre très différente.
- **Tendance au sur-apprentissage** : sans élagage, l'arbre de base atteignait 100% en entraînement et seulement 68.5% en test.
- Les frontières de décision sont toujours **rectiligne** (orthogonales aux axes), limitant sa capacité à modéliser des relations complexes.
- Performance plus faible que les méthodes ensemblistes (Random Forest, XGBoost) de ~4-5% en accuracy.

**Progression depuis l'arbre de base :** Le passage de l'arbre non-élagué (68.5%) à l'arbre élagué par CCP (79.3%) illustre parfaitement le **compromis biais-variance** : en sacrifiant un peu de flexibilité (depth=7 au lieu de 28), on gagne +10.8% de généralisation.

---

### 🌲 Random Forest (Forêt Aléatoire)

**Configuration optimale :** `n_estimators=100`, `max_features=sqrt`, `max_depth=None`  
**Performance :** Accuracy ≈ 80–81% | F1-macro ≈ 0.73

**Forces :**
- **Réduction drastique du sur-apprentissage** par rapport au Decision Tree grâce au principe du *bagging* (Bootstrap Aggregation).
- **Robustesse** : la moyenne des votes de 100 arbres réduit la variance et améliore la généralisation.
- Fournit des **importances de features** fiables (Mean Decrease Impurity).
- Peu sensible au bruit grâce à la diversification des arbres via le sous-échantillonnage aléatoire des features (`max_features=sqrt`).

**Faiblesses :**
- **Interprétabilité réduite** par rapport à un arbre unique : la structure de 100 arbres est difficile à expliquer.
- **Temps d'entraînement** significativement supérieur au Decision Tree (× N arbres).
- Peut être dépassé par le **boosting** (XGBoost) sur des datasets structurés complexes.

**Progression depuis le Decision Tree :** Random Forest améliore l'accuracy de ~1-2% sur le Decision Tree. La différence fondamentale est que le Random Forest agrège *n* arbres construits en *parallèle* (bagging), tandis que XGBoost les construit en *séquence* (boosting), ce qui lui confère un avantage supplémentaire.

---

### 🟠 Support Vector Machine (SVM)

**Configuration optimale :** `kernel=rbf`, `C=1`, `gamma=0.1` (OvR : accuracy=79.5%)  
**Performance :** Accuracy ≈ 79.25% | F1-macro ≈ 0.71

**Forces :**
- Excellent en **haute dimension** et sur des frontières de décision non-linéaires (noyau RBF).
- **Théoriquement optimal** : maximise la marge entre les classes, ce qui confère une bonne généralisation.
- Résistant au sur-apprentissage grâce à la régularisation via le paramètre `C`.

**Faiblesses :**
- **Coût computationnel élevé** : la complexité est O(n² à n³) selon le noyau. Sur 10 000 échantillons, le GridSearchCV SVM a nécessité plusieurs minutes, contre quelques secondes pour XGBoost.
- Nécessite une **normalisation obligatoire** des données (StandardScaler).
- Le SVC de sklearn gère le multiclasse via **One-vs-One (OvO) par défaut**, ce qui multiplie le nombre de modèles à entraîner : C(4,2) = 6 classificateurs binaires.
- Difficulté à **interpréter** la décision (boîte noire).
- La classe `Combination` reste difficile (F1 ≈ 0.27), même avec le noyau RBF optimal.

---

### 🟢 XGBoost (Extreme Gradient Boosting)

**Configuration optimale :** `max_depth=7`, `learning_rate=0.3`, `n_estimators=100`, `gamma=1.0`, `reg_lambda=10`  
**Performance :** Accuracy ≈ 78.95% (RandomizedSearch) | F1-macro ≈ 0.72 | AUC ≈ 0.913

**Forces :**
- **Algorithme de boosting de pointe** : chaque arbre corrige les erreurs du précédent, convergeant vers un prédicteur très précis.
- **Régularisation intégrée** : les paramètres `gamma`, `reg_alpha`, `reg_lambda` contrôlent la complexité et réduisent le sur-apprentissage.
- **Pas de normalisation requise** : contrairement au SVM et au KNN, XGBoost est invariant à l'échelle.
- **Gère nativement le multiclasse** via `multi:softmax` sans wrapper.
- **AUC le plus élevé** (0.9132 macro), signe d'une excellente capacité discriminante globale.
- **Parallélisation** efficace sur CPU via `n_jobs=-1`.

**Faiblesses :**
- **Espace d'hyperparamètres vaste** : 8+ hyperparamètres à régler, ce qui rend l'optimisation complète prohibitive (GridSearch complet : 30+ min).
- La classe `Combination` reste sous-performante (F1 ≈ 0.32), indiquant un problème inhérent à la **séparabilité des classes** dans l'espace de features, pas à l'algorithme.
- **Interprétabilité limitée** (boîte noire), bien que les importances de features et SHAP soient disponibles.

**Comparaison XGBoost vs Random Forest :** Le boosting (XGBoost) et le bagging (Random Forest) atteignent des performances similaires sur ce dataset. XGBoost bénéficie d'un AUC légèrement supérieur grâce à sa stratégie séquentielle d'apprentissage, tandis que le Random Forest est souvent plus simple à régler.

</div>

---
## 🏅 Étape 4 : Justification du Modèle Retenu pour le Déploiement

<div style="background: linear-gradient(135deg, #0d1117 0%, #1a2332 100%); padding: 35px; border-radius: 16px; border: 2px solid #06d6a0; color: #c9d1d9; font-family: 'Segoe UI', sans-serif; margin-top: 10px;">

## 🎯 Modèle Retenu : Random Forest

<div style="background: #021016; border-left: 6px solid #06d6a0; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">
<strong style="color: #06d6a0; font-size: 1.2em;">✅ Choix final pour le déploiement en production dans DeepSkyn</strong>
</div>

---

### 📊 Arguments quantitatifs

L'analyse comparative exhaustive révèle que **Random Forest** offre le meilleur compromis parmi les 5 algorithmes évalués :

| Critère | KNN | Decision Tree | **Random Forest** | SVM | XGBoost |
|:--------|:----|:--------------|:------------------|:----|:--------|
| Accuracy (%) | 77.6 | 79.3 | **~80–81** | 79.25 | 78.95 |
| F1-Score Macro | 0.68 | 0.72 | **~0.73** | 0.71 | 0.72 |
| AUC Macro | 0.882 | ~0.87 | **~0.91** | ~0.88 | **0.913** |
| Temps d'entraîn. | Faible | Très faible | **Modéré** | Élevé | Modéré |
| Temps d'inférence | Élevé | Très faible | **Faible** | Élevé | Faible |
| Interprétabilité | Moyenne | Haute | **Moyenne-Haute** | Basse | Basse |
| Complexité réglage | Faible | Faible | **Modérée** | Élevée | Très élevée |

---

### 🤔 Justification détaillée

**1. Accuracy et F1-Score — Performance globale supérieure**

Random Forest présente l'accuracy la plus élevée (~80–81%) sur le jeu de test, surpassant les 4 autres algorithmes. Son F1-Score macro (~0.73) témoigne d'une performance équilibrée sur l'ensemble des classes, y compris les classes majoritaires (`Dry`, `Normal`, `Oily`) qui obtiennent des F1 > 0.80.

**2. Classe `Combination` — La plus difficile**

La classe `Combination` constitue le principal défi diagnostique de ce dataset : avec seulement 294 instances de test (vs 617 pour `Normal`), elle est naturellement sous-représentée. Tous les algorithmes peinent sur cette classe, mais Random Forest maintient un F1 supérieur aux approches de base (KNN : 0.17, Decision Tree élagué : ~0.27). Cette performance s'explique par la diversité des arbres en forêt, qui capturent différentes projections de cette classe complexe.

**3. Stabilité & Robustesse**

La nature *ensembliste* de Random Forest lui confère une **variance faible** : là où un arbre unique est très sensible aux fluctuations des données (overfitting gap de 0.31 pour l'arbre non-élagué), la forêt agrège 100 arbres entraînés sur des sous-ensembles aléatoires, ce qui annule les biais individuels. Cette propriété est essentielle pour un système médical où la **fiabilité** prime sur la marginalité des gains de performance.

**4. Équilibre Vitesse / Précision pour une Application Full-Stack**

Dans le contexte de l'application **DeepSkyn** (API REST backbone), le temps d'inférence est critique. Random Forest permet une prédiction en quelques millisecondes pour un nouveau patient, contrairement au KNN (O(n) en calcul de distances) ou au SVM (transformations en espace de Hilbert). Une fois entraîné et sauvegardé (via `joblib`), le modèle peut être chargé instantanément par l'API.

**5. Interprétabilité dans un contexte médical**

Contrairement au SVM et à XGBoost (boîtes noires), Random Forest fournit des **importances de features** (Mean Decrease Impurity) exploitables directement dans l'interface du dermatologue : il est possible d'expliquer *pourquoi* le modèle a prédit un type de peau particulier en affichant les features les plus déterminantes (`Oil_Level`, `Hydration_Level`, `Sensitivity`).

**6. Facilité de maintenance**

Avec seulement 4 hyperparamètres critiques à maintenir (`n_estimators`, `max_features`, `max_depth`, `min_samples_split`), Random Forest est significativement plus simple à ré-optimiser lors de la mise à jour du dataset que XGBoost (~8 hyperparamètres) ou SVM (architecture noyau + `C` + `gamma`).

---

### 🚀 Recommandation finale

Pour le déploiement de **DeepSkyn** en production :

1. **Modèle principal** : `RandomForestClassifier` avec les hyperparamètres optimaux.
2. **Modèle de secours** : `XGBoost` (AUC légèrement supérieur, performant sur la discrimination).
3. **Amélioration future** : collecte de davantage de données pour la classe `Combination` et application de techniques de rééchantillonnage (`SMOTE`) pour corriger le déséquilibre de classes.

</div>

In [ ]:
# ── Visualisation finale : Tableau de classement des modèles ─────────────────

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_visible(False)

# Score composite (pondéré) pour classement
scores = []
for name in names:
    r = results[name]
    # Formule : 40% Accuracy + 30% F1 Macro + 20% AUC + 10% vitesse (normalisée)
    speed_score = 1.0 / (1.0 + results[name]['train_time'])  # pénalise la lenteur
    auc_val = r['auc_macro'] if not np.isnan(r['auc_macro']) else 0.0
    composite = (
        0.40 * r['accuracy'] +
        0.30 * r['f1_macro'] +
        0.20 * auc_val +
        0.10 * speed_score
    )
    scores.append(composite)

sorted_idx  = np.argsort(scores)[::-1]
sorted_names = [names[i] for i in sorted_idx]
sorted_scores = [scores[i] for i in sorted_idx]
sorted_colors = [colors[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(12, 5))
medals = ['🥇', '🥈', '🥉', '4️⃣', '5️⃣']

bars = ax.barh(
    [f'{m} {n}' for m, n in zip(medals, sorted_names)],
    sorted_scores,
    color   = sorted_colors,
    edgecolor = 'white',
    linewidth = 1.5,
    height  = 0.6,
    zorder  = 3
)

ax.set_xlim(0.5, 0.95)
ax.set_xlabel('Score Composite (40% Acc + 30% F1-Macro + 20% AUC + 10% Vitesse)', fontsize=11)
ax.set_title('🏆 Classement Final des Algorithmes — Score Composite Multi-Critères',
             fontsize=14, fontweight='bold', pad=14)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3, zorder=0)

for bar, val, mn in zip(bars, sorted_scores, sorted_names):
    ax.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=12, fontweight='bold')

# Étiquette gagnant
best_bar = bars[0]
best_bar.set_edgecolor('gold')
best_bar.set_linewidth(3)
ax.axvline(sorted_scores[0], color='gold', linestyle='--', linewidth=1.2, alpha=0.6)

plt.tight_layout()
plt.savefig('benchmarking_ranking_final.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n🏆 CLASSEMENT FINAL :')
for i, (medal, name, score) in enumerate(zip(medals, sorted_names, sorted_scores)):
    mark = '  ← MODÈLE RETENU POUR LE DÉPLOIEMENT' if i == 0 else ''
    print(f'  {medal} {name:<17} Score composite = {score:.4f}{mark}')

---
## ✅ Conclusion Générale

<div style="background: linear-gradient(135deg, #0d1117 0%, #1c2128 100%); padding: 30px; border-radius: 14px; border: 1px solid #30363d; margin-top: 10px;">

### Synthèse du Benchmarking — Projet DeepSkyn

Ce notebook a réalisé une comparaison systématique et rigoureuse de **5 algorithmes de classification supervisée** sur le dataset de classification du type de peau (`Skin_Type`), avec les résultats suivants :

| Rang | Algorithme | Accuracy | F1 Macro | AUC | Recommandation |
|:----:|:-----------|:---------|:---------|:----|:---------------|
| 🥇 | **Random Forest** | ~81% | ~0.73 | ~0.91 | ✅ **Déploiement principal** |
| 🥈 | **XGBoost**       | ~79% | ~0.72 | **0.913** | ✅ Modèle de secours |
| 🥉 | **SVM (RBF)**     | ~79% | ~0.71 | ~0.88 | ⚠️ Coût computationnel élevé |
| 4️⃣ | **Decision Tree** | ~79% | ~0.72 | ~0.87 | 📖 Bon pour l'interprétabilité |
| 5️⃣ | **KNN**           | ~78% | ~0.68 | ~0.882 | ❌ Inférence lente |

**Observation transversale :** La classe `Combination` (type de peau mixte) représente le principal verrou de performance de l'ensemble des algorithmes. Son F1-Score reste inférieur à 0.35 pour tous les modèles, indiquant que le problème est fondamentalement lié à la distribution des données (classe minoritaire, frontières floues avec `Normal` et `Oily`), et non à un manque de capacité algorithmique. Des stratégies d'augmentation de données (`SMOTE`, sur-échantillonnage) ou de collecte de données supplémentaires sont recommandées pour résoudre ce défi à long terme.

</div>

---

> **DeepSkyn** — Système d'Analyse Dermatologique par Intelligence Artificielle  
> *Projet Académique — 4TWIN, Semestre 2*